In [ ]:
# mobilenet_ip102_debug.py

import torch
import torch.nn as nn
import time
import numpy as np
import os
from PIL import Image
from torchvision import transforms, models
from torch.utils.data import Dataset, DataLoader

# -----------------------
# Paths
# -----------------------

DATA_ROOT = "ip102"

TRAIN_TXT = os.path.join(DATA_ROOT, "train.txt")
VAL_TXT   = os.path.join(DATA_ROOT, "val.txt")

TRAIN_DIR = os.path.join(DATA_ROOT, "classification/train")
VAL_DIR   = os.path.join(DATA_ROOT, "classification/val")

NUM_CLASSES = 102
BATCH_SIZE = 32
EPOCHS = 5
WORKERS = 2

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print("Device:", device)
print("Train TXT:", TRAIN_TXT)
print("Val TXT:", VAL_TXT)
print("Train DIR:", TRAIN_DIR)
print("Val DIR:", VAL_DIR)

# -----------------------
# Transforms
# -----------------------

transform = transforms.Compose([
    transforms.Resize((224,224)),
    transforms.RandomHorizontalFlip(),
    transforms.ToTensor(),
    transforms.Normalize(
        [0.485,0.456,0.406],
        [0.229,0.224,0.225]
    )
])

# -----------------------
# Dataset
# -----------------------

class IP102Dataset(Dataset):

    def __init__(self, txt_file, img_root, transform=None):

        print("\nLoading dataset:", txt_file)

        self.samples = []
        self.img_root = img_root
        self.transform = transform

        with open(txt_file) as f:
            lines = f.readlines()

        print("Lines found:", len(lines))

        for i, line in enumerate(lines):

            img_name, label = line.strip().split()
            label = int(label)

            img_path = os.path.join(
                img_root,
                str(label),
                img_name
            )

            if i < 5:
                print("Example path:", img_path)
                print("Exists:", os.path.exists(img_path))

            self.samples.append((img_path, label))

        print("Dataset loaded:", len(self.samples))

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):

        img_path, label = self.samples[idx]

        if not os.path.exists(img_path):
            print("Missing file:", img_path)
            raise FileNotFoundError(img_path)

        image = Image.open(img_path).convert("RGB")

        if self.transform:
            image = self.transform(image)

        return image, label


# -----------------------
# Load datasets
# -----------------------

train_ds = IP102Dataset(TRAIN_TXT, TRAIN_DIR, transform)
val_ds   = IP102Dataset(VAL_TXT, VAL_DIR, transform)

print("\nTrain samples:", len(train_ds))
print("Val samples:", len(val_ds))

train_loader = DataLoader(
    train_ds,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=WORKERS,
    pin_memory=True
)

val_loader = DataLoader(
    val_ds,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=WORKERS,
    pin_memory=True
)

print("DataLoaders created")

# -----------------------
# Model
# -----------------------

print("\nLoading model...")

model = models.mobilenet_v2(
    weights=models.MobileNet_V2_Weights.DEFAULT
)

model.classifier[1] = nn.Linear(
    model.last_channel,
    NUM_CLASSES
)

model = model.to(device)

print("Model ready")

# -----------------------
# Training setup
# -----------------------

criterion = nn.CrossEntropyLoss()

optimizer = torch.optim.SGD(
    model.parameters(),
    lr=0.01,
    momentum=0.9
)

scheduler = torch.optim.lr_scheduler.StepLR(
    optimizer,
    step_size=7,
    gamma=0.1
)

# -----------------------
# Training loop
# -----------------------

print("\nStarting training...")

for epoch in range(EPOCHS):

    model.train()
    running_loss = 0
    t0 = time.time()

    for batch_idx, (imgs, labels) in enumerate(train_loader):

        if batch_idx % 50 == 0:
            print(f"Epoch {epoch+1} Batch {batch_idx}")

        imgs = imgs.to(device)
        labels = labels.to(device)

        optimizer.zero_grad()

        outputs = model(imgs)

        loss = criterion(outputs, labels)

        loss.backward()

        optimizer.step()

        running_loss += loss.item()

    scheduler.step()

    print(
        f"Epoch {epoch+1}/{EPOCHS} | "
        f"loss={running_loss:.4f} | "
        f"time={time.time()-t0:.1f}s"
    )

# -----------------------
# Evaluation
# -----------------------

print("\nEvaluating...")

def evaluate(model, loader):

    model.eval()

    preds_all = []
    labels_all = []

    with torch.no_grad():

        for i, (imgs, labels) in enumerate(loader):

            if i % 20 == 0:
                print("Eval batch:", i)

            imgs = imgs.to(device)

            outputs = model(imgs)

            preds = outputs.argmax(dim=1).cpu().numpy()

            preds_all.extend(preds)
            labels_all.extend(labels.numpy())

    acc = np.mean(
        np.array(preds_all) == np.array(labels_all)
    )

    return acc

val_acc = evaluate(model, val_loader)

print("Validation accuracy:", val_acc)

# -----------------------
# Model size
# -----------------------

param_count = sum(p.numel() for p in model.parameters())

print("Parameters:", f"{param_count:,}")

# -----------------------
# Inference speed test
# -----------------------

print("\nTesting inference speed...")

model.eval()

dummy = torch.randn(1,3,224,224).to(device)

for _ in range(10):
    model(dummy)

N = 100

t0 = time.time()

for _ in range(N):
    model(dummy)

avg = (time.time()-t0)/N

print(f"Inference time: {avg*1000:.2f} ms")